# Hledání A* – od her po průmysl

V notebooku P vs NP jsme viděli, že *nejkratší cesta* je v P – ale „polynomiální“ neznamená „zadarmo“:
hra s 500 jednotkami nebo sklad s 200 roboty přeplánovává tisíce cest za sekundu a silniční síť velikosti státu
má $10^7$ křižovatek. A* je způsob, jak P problém udělat v praxi dost rychlým: Dijkstrův algoritmus plus **heuristika**.

V prvním cvičení jsme viděli A* na hračkovém grafu s pěti uzly. Tady ho použijeme na úlohy, kde na něm opravdu záleží:

1. **Hra:** hledání cesty na mapě z dlaždic (stejný algoritmus pohybuje jednotkami ve StarCraftu, Age of Empires, Factoriu, ...).
2. **Hra:** 8-puzzle – A* ve stavovém prostoru bez jakékoli mapy.
3. **Průmysl:** vychystávání zboží ve skladu – A* spočítá vzdálenosti, TSP určí pořadí.
4. **Průmysl:** navigace na silniční síti – jak GPS hledá trasu.

Klíčová myšlenka A*: vždy pokračuj z kandidáta s nejmenším

$$f(n) = g(n) + h(n)$$

kde $g(n)$ je cena už zaplacená od startu a $h(n)$ je **heuristika** – odhad ceny, která ještě zbývá.
Pokud $h$ nikdy nepřecení (je *přípustná*), A* najde optimální cestu – a čím lepší odhad, tím méně uzlů musí prohlédnout.

*Jak číst kód:* čtěte komentáře; přesná syntaxe Pythonu je vedlejší.

In [ ]:
import heapq                       # "prioritní fronta": hromádka, ze které vždy rychle vezmeme nejmenší prvek
import math                        # odmocniny, nekonečno
import itertools                   # kombinace / permutace
from time import perf_counter      # stopky

import numpy as np                 # tabulky čísel (mapy jsou 2-D tabulky)
import pandas as pd                # hezké tabulky výsledků
import matplotlib.pyplot as plt    # obrázky
import networkx as nx              # grafy (silniční síť)

## Obecná implementace A*

Funkci je jedno, jestli *uzel* je políčko mřížky, rozložení hlavolamu nebo křižovatka –
potřebuje jen funkci `neighbors(uzel)`, která vrací dvojice `(soused, cena_kroku)`, a heuristiku `h(uzel)`.

* `heuristic = lambda n: 0` udělá z A* **Dijkstrův algoritmus** (prohledávání podle ceny),
* `weight > 1` dá **vážené A*** – rychlejší, ale cesta může být až `weight`× delší než optimum.

In [ ]:
def a_star(start, goal, neighbors, heuristic, weight: float = 1.0):
    """
    Obecné hledání A*.

    Parametry:
        start, goal : dva uzly (cokoli, co lze porovnávat a použít jako klíč slovníku)
        neighbors   : funkce uzel -> seznam (soused, cena_kroku)   ("kam odsud můžu a co to stojí")
        heuristic   : funkce uzel -> odhad ceny do cíle             ("jak je to ještě zhruba daleko")
        weight      : násobitel heuristiky (1 = klasické A*, 0 = Dijkstra, >1 = vážené A*)

    Vrací:
        path (seznam uzlů), cost (číslo), expanded (množina uzlů, které jsme "otevřeli" = vytáhli z fronty)
    """
    # Fronta kandidátů. Každý záznam je (f, g, uzel); heapq nám vždy dá záznam s nejmenším f.
    open_set = [(weight * heuristic(start), 0.0, start)]
    came_from = {start: None}      # pro každý uzel: z kterého uzlu jsme do něj přišli (pro sestavení cesty na konci)
    g = {start: 0.0}               # pro každý uzel: nejlevnější známá cena od startu
    expanded = set()               # uzly, které jsme už zpracovali ("uzavřená množina")

    while open_set:                                        # dokud jsou kandidáti
        _, g_current, current = heapq.heappop(open_set)    # vezmi nejslibnějšího (nejmenší f)
        if current in expanded:                            # zastaralý duplicitní záznam -> ignoruj
            continue
        expanded.add(current)                              # označ jako zpracovaný

        if current == goal:                                # dosáhli jsme cíle -> sestav cestu pozpátku
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return path[::-1], g_current, expanded         # otoč, aby začínala ve `start`

        for neighbor, step_cost in neighbors(current):     # projdi všechny sousedy aktuálního uzlu
            if neighbor in expanded:                       # už zpracovaný -> nic lepšího nepřijde
                continue
            g_new = g_current + step_cost                  # cena dosažení souseda přes aktuální uzel
            if g_new < g.get(neighbor, math.inf):          # levnější než cokoli dosud známého?
                g[neighbor] = g_new                        # zapamatuj lepší cenu ...
                came_from[neighbor] = current              # ... a odkud přišla
                heapq.heappush(open_set, (g_new + weight * heuristic(neighbor), g_new, neighbor))   # přidej mezi kandidáty

    return None, math.inf, expanded                        # fronta prázdná a cíl nedosažen -> cesta neexistuje

## 1. Hra: hledání cesty na mapě z dlaždic

Jednotka musí dojít ze startu (zelená) do cíle (zlatá). Zdi jsou černé.
Mapa obsahuje klasickou past – zeď ve tvaru U otevřenou směrem ke startu.

In [ ]:
def make_game_map(rows: int = 25, cols: int = 40, seed: int = 7) -> np.ndarray:
    """2-D tabulka dlaždic: 0 = volno, 1 = zeď."""
    grid = np.zeros((rows, cols), dtype=int)
    rng = np.random.default_rng(seed)             # generátor náhody s pevným seedem -> pokaždé stejná mapa

    # pár náhodných kamenů: zhruba 12 % dlaždic se stane zdí
    rocks = rng.random((rows, cols)) < 0.12
    grid[rocks] = 1

    # past ve tvaru U (otevřená doleva, tj. směrem ke startu)
    grid[5:20, 22] = 1          # svislá zeď (dno U)
    grid[5, 12:23] = 1          # horní rameno
    grid[19, 12:23] = 1         # dolní rameno
    grid[6:19, 13:22] = 0       # vnitřek U nech volný

    return grid


def grid_neighbors_4(grid: np.ndarray):
    """Vrátí funkci sousedů pro pohyb ve 4 směrech (nahoru/dolů/vlevo/vpravo) s cenou 1 za krok."""
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):        # čtyři směry
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:   # uvnitř mapy a není to zeď
                yield (nr, nc), 1
    return neighbors


def show_search(grid, start, goal, path, expanded, title, ax=None):
    """Nakreslí mapu: zdi černě, prohlédnuté dlaždice světle modře, cestu červeně, start zeleně, cíl zlatě."""
    ax = ax or plt.gca()
    image = np.ones(grid.shape + (3,))            # RGB obrázek, všude bílá
    image[grid == 1] = (0.1, 0.1, 0.1)            # zdi
    for cell in expanded:
        image[cell] = (0.7, 0.85, 1.0)            # dlaždice, na které se algoritmus podíval
    if path:
        for cell in path:
            image[cell] = (0.9, 0.2, 0.2)         # nalezená cesta
    image[start] = (0.1, 0.7, 0.1)
    image[goal] = (1.0, 0.8, 0.0)
    ax.imshow(image, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


game_map = make_game_map()
start, goal = (12, 3), (12, 36)                  # (řádek, sloupec)
game_map[start] = game_map[goal] = 0             # pojistka: start a cíl nejsou zeď

plt.figure(figsize=(8, 5))
show_search(game_map, start, goal, [], set(), "Mapa")
plt.show()

### Která heuristika? Porovnejme počet prohlédnutých dlaždic

Všechny heuristiky níže jsou pro pohyb ve 4 směrech přípustné, ale manhattanská vzdálenost je nejtěsnější odhad.
„Greedy best-first“ používá (skoro) jen heuristiku – je rychlé, ale U-past ho pošle na dlouhou objížďku.

In [ ]:
# Odhady vzdálenosti mezi dlaždicemi a = (řádek, sloupec) a b = (řádek, sloupec):
def manhattan(a, b): return abs(a[0] - b[0]) + abs(a[1] - b[1])      # řádky + sloupce  (na prázdné mřížce se 4 směry přesné)
def euclidean(a, b): return math.hypot(a[0] - b[0], a[1] - b[1])     # vzdušnou čarou   (vždy <= Manhattan)
def chebyshev(a, b): return max(abs(a[0] - b[0]), abs(a[1] - b[1]))  # větší z obou     (vždy <= Euklid)
def zero(a, b):      return 0                                        # žádný odhad = Dijkstra

heuristics = {
    "Dijkstra (h = 0)":      (zero, 1),
    "A* Čebyšev":            (chebyshev, 1),
    "A* Euklid":             (euclidean, 1),
    "A* Manhattan":          (manhattan, 1),
    "Vážené A* (w = 3)":     (manhattan, 3),        # f = g + 3h
    "Greedy best-first":     (manhattan, 1000),     # f ≈ jen h
}

neighbors = grid_neighbors_4(game_map)
results, runs = [], {}
for name, (h, w) in heuristics.items():
    t0 = perf_counter()
    # lambda n, h=h: h(n, goal)  -> "heuristika dlaždice n je její odhad vzdálenosti do cíle"
    path, cost, expanded = a_star(start, goal, neighbors, lambda n, h=h: h(n, goal), weight=w)
    runs[name] = (path, expanded)
    results.append({"algoritmus": name, "prohlédnuté dlaždice": len(expanded), "délka cesty": cost,
                    "čas [ms]": 1000 * (perf_counter() - t0)})

pd.DataFrame(results).set_index("algoritmus")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, (name, (path, expanded)) in zip(axes.flat, runs.items()):
    show_search(game_map, start, goal, path, expanded, f"{name}\nprohlédnuto: {len(expanded)}, délka: {len(path) - 1}", ax)
plt.tight_layout()
plt.show()

### Vážené A*: optimalita výměnou za rychlost

Ve hrách často nepotřebujeme *tu* nejkratší cestu – cesta o 10 % delší, ale nalezená 5× rychleji, je skvělý obchod,
když 500 jednotek přeplánovává každý snímek. Vážené A* s $f = g + w \cdot h$ zaručuje cestu nejvýš $w$× delší než optimum.

In [ ]:
rows = []
for w in [1, 1.2, 1.5, 2, 3, 5, 10]:
    path, cost, expanded = a_star(start, goal, neighbors, lambda n: manhattan(n, goal), weight=w)
    rows.append({"váha w": w, "prohlédnuté dlaždice": len(expanded), "délka cesty": cost})
weighted = pd.DataFrame(rows).set_index("váha w")
weighted["% delší než optimum"] = 100 * (weighted["délka cesty"] / weighted["délka cesty"].iloc[0] - 1)
weighted

## 2. Hra: 8-puzzle

Tady není žádná mapa – *stav* je rozložení dlaždic a *tah* posune jednu dlaždici do mezery.
Stavový prostor má $9!/2 = 181\,440$ dosažitelných rozložení. Vyřešíme nejtěžší možnou instanci (31 tahů)
třemi heuristikami a spočítáme prohlédnuté stavy.

* `h = 0` – prohledávání do šířky (každý tah stojí 1, takže Dijkstra = BFS),
* *špatně umístěné dlaždice* – počet dlaždic mimo cílovou pozici,
* *manhattanská vzdálenost* – součet vzdáleností každé dlaždice od její cílové pozice.

In [ ]:
# Stav je n-tice 9 čísel čtená po řádcích; 0 = mezera.
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)      # 1 2 3 / 4 5 6 / 7 8 _
HARDEST = (8, 6, 7, 2, 5, 4, 3, 0, 1)   # potřebuje 31 tahů


def puzzle_neighbors(state):
    """Všechny stavy dosažitelné jedním tahem (posunutí sousední dlaždice do mezery). Každý tah stojí 1."""
    i = state.index(0)                  # pozice mezery (0..8)
    r, c = divmod(i, 3)                 # její řádek a sloupec
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):       # dlaždice pod / nad / vpravo / vlevo od mezery
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:                     # musí být uvnitř desky 3x3
            j = nr * 3 + nc
            s = list(state)
            s[i], s[j] = s[j], s[i]                         # prohoď dlaždici a mezeru
            yield tuple(s), 1


def h_misplaced(state):
    """Počet dlaždic, které nejsou tam, kde mají být (mezera se nepočítá)."""
    return sum(1 for i, tile in enumerate(state) if tile and tile != i + 1)


def h_manhattan(state):
    """Součet přes dlaždice: (řádky + sloupce) mezi aktuální a cílovou pozicí dlaždice."""
    return sum(abs(i // 3 - (tile - 1) // 3) + abs(i % 3 - (tile - 1) % 3)
               for i, tile in enumerate(state) if tile)


def show_state(state):
    """Vypíše desku jako 3 řádky."""
    return "\n".join(" ".join(str(t) if t else "." for t in state[r * 3:(r + 1) * 3]) for r in range(3))


print(show_state(HARDEST), "\n")

rows = []
for name, h in [("BFS (h = 0)", lambda s: 0), ("A* špatně umístěné", h_misplaced), ("A* Manhattan", h_manhattan)]:
    t0 = perf_counter()
    path, cost, expanded = a_star(HARDEST, GOAL, puzzle_neighbors, h)
    rows.append({"algoritmus": name, "prohlédnuté stavy": len(expanded), "tahů": int(cost),
                 "čas [s]": perf_counter() - t0})
pd.DataFrame(rows).set_index("algoritmus")

Všechny tři najdou optimální řešení na 31 tahů – ale dobrá heuristika prohlédne stokrát méně stavů.
Stejný trik (A* / IDA* s heuristikou z *pattern database*) řeší optimálně 15-puzzle i Rubikovu kostku.

## 3. Průmysl: vychystávání ve skladu (A* + TSP)

Skladník (nebo robot AGV) vyjíždí z doku, má posbírat 6 položek z regálů a vrátit se.
Kolem balicí zóny jsou uličky přeplněné, krok tam stojí 3× víc.

Přesně tuto úlohu řeší skladový systém tisíckrát denně a spojuje oba algoritmy tohoto cvičení:

1. **A\*** spočítá vzdálenost chůze mezi každou dvojicí míst (mapa má překážky, vzdušná čára nestačí),
2. **TSP** rozhodne, v jakém pořadí místa navštívit.

In [ ]:
def make_warehouse(rows: int = 22, cols: int = 31):
    """Mapa skladu (0 = ulička, 1 = regál) a cena vstupu na každou dlaždici."""
    grid = np.zeros((rows, cols), dtype=int)
    for c in (3, 4, 8, 9, 13, 14, 18, 19, 23, 24):           # regály jsou svislé bloky široké dva sloupce
        grid[3:19, c] = 1
    cost = np.ones((rows, cols))                             # normálně krok stojí 1 ...
    cost[rows - 3:, :11] = 3                                 # ... ale 3 v přeplněné balicí zóně (vlevo dole)
    return grid, cost


def warehouse_neighbors(grid, cost):
    """Funkce sousedů: 4 směry, cena kroku je cena dlaždice, na kterou vstupujeme."""
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 0:
                yield (nr, nc), cost[nr, nc]
    return neighbors


warehouse, step_cost = make_warehouse()
dock = (20, 1)
picks = [(5, 5), (16, 7), (9, 12), (14, 15), (6, 20), (17, 25)]     # (řádek, sloupec) šesti položek
locations = [dock] + picks
labels = ["dok"] + [f"P{i + 1}" for i in range(len(picks))]


def show_warehouse(path=None, title="Sklad"):
    image = np.ones(warehouse.shape + (3,))
    image[step_cost == 3] = (1.0, 0.9, 0.8)       # přeplněná zóna: oranžová
    image[warehouse == 1] = (0.35, 0.25, 0.15)    # regály: hnědá
    if path:
        for cell in path:
            image[cell] = (0.9, 0.2, 0.2)         # trasa: červená
    plt.figure(figsize=(9, 6))
    plt.imshow(image, interpolation="nearest")
    for label, (r, c) in zip(labels, locations):
        plt.plot(c, r, "o", color="gold" if label == "dok" else "royalblue", markersize=14)
        plt.text(c, r, label, ha="center", va="center", fontsize=7, fontweight="bold")
    plt.title(title); plt.xticks([]); plt.yticks([])
    plt.show()


show_warehouse(title="Sklad: regály (hnědě), přeplněná balicí zóna (oranžově), místa položek (modře)")

In [ ]:
# Krok 1: vzdálenosti mezi všemi dvojicemi pomocí A*  (Manhattan je přípustný, protože každý krok stojí aspoň 1)
neighbors = warehouse_neighbors(warehouse, step_cost)
k = len(locations)
distance_matrix = np.zeros((k, k))
paths = {}
for i, j in itertools.combinations(range(k), 2):          # každá dvojice míst (21 dvojic)
    path, cost, _ = a_star(locations[i], locations[j], neighbors, lambda n, g=locations[j]: manhattan(n, g))
    distance_matrix[i, j] = distance_matrix[j, i] = cost  # vzdálenost chůze do matice (oběma směry)
    paths[i, j] = path                                    # skutečnou cestu si necháme pro vykreslení trasy
    paths[j, i] = path[::-1]

pd.DataFrame(distance_matrix, index=labels, columns=labels)

In [ ]:
# Krok 2: pořadí položek = TSP na matici vzdáleností (6 položek -> 720 permutací, hrubá síla stačí)
def tour_length(order):
    """Délka trasy dok -> pořadí položek -> dok."""
    tour = [0] + list(order) + [0]
    return sum(distance_matrix[a, b] for a, b in zip(tour, tour[1:]))

best_order = min(itertools.permutations(range(1, k)), key=tour_length)     # zkus všech 720 pořadí, nech nejkratší

# pořadí podle nejbližšího souseda pro srovnání
remaining, current, greedy_order = set(range(1, k)), 0, []
while remaining:
    current = min(remaining, key=lambda j: distance_matrix[current, j])     # nejbližší nenavštívená položka
    remaining.remove(current)
    greedy_order.append(current)

print("Optimální pořadí:", " -> ".join(labels[i] for i in [0, *best_order, 0]), f"  délka {tour_length(best_order):.0f}")
print("Nejbližší soused:", " -> ".join(labels[i] for i in [0, *greedy_order, 0]), f"  délka {tour_length(greedy_order):.0f}")

# Krok 3: slep cesty z A* dohromady a nakresli trasu
tour = [0] + list(best_order) + [0]
full_route = [cell for a, b in zip(tour, tour[1:]) for cell in paths[a, b]]
show_warehouse(full_route, title=f"Optimální vychystávací trasa, délka {tour_length(best_order):.0f}")

## 4. Průmysl: navigace na silniční síti

Silniční mapa je graf se souřadnicemi: křižovatky jsou uzly, silnice jsou hrany ohodnocené délkou.
Dijkstrův algoritmus prohledává do všech stran jako rostoucí kruh; A* s heuristikou vzdušné čáry
prohledává hlavně *směrem* k cíli.

(Skutečné navigace přidávají další triky – contraction hierarchies, landmarky – ale jádro je A*.)

In [ ]:
# Náhodná "silniční síť": 600 křižovatek, silnice mezi křižovatkami blíž než 0,075
roads = nx.random_geometric_graph(600, 0.075, seed=4)
roads = roads.subgraph(max(nx.connected_components(roads), key=len)).copy()   # nech jen souvislou část
coordinates = nx.get_node_attributes(roads, "pos")
for u, v in roads.edges():
    roads[u][v]["weight"] = math.dist(coordinates[u], coordinates[v])         # délka silnice = vzdálenost vzdušnou čarou

origin = min(roads, key=lambda n: math.dist(coordinates[n], (0.05, 0.05)))        # křižovatka u levého dolního rohu
destination = min(roads, key=lambda n: math.dist(coordinates[n], (0.95, 0.95)))   # křižovatka u pravého horního rohu


def road_neighbors(u):
    """Sousední křižovatky a délka silnice k nim."""
    return ((v, roads[u][v]["weight"]) for v in roads[u])


def straight_line(u):
    """Heuristika: vzdálenost vzdušnou čarou do cíle (nikdy není delší než skutečná vzdálenost po silnicích)."""
    return math.dist(coordinates[u], coordinates[destination])


fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, (name, h) in zip(axes, [("Dijkstra", lambda n: 0), ("A* (heuristika vzdušnou čarou)", straight_line)]):
    t0 = perf_counter()
    path, cost, expanded = a_star(origin, destination, road_neighbors, h)
    elapsed = perf_counter() - t0
    nx.draw_networkx_edges(roads, coordinates, ax=ax, edge_color="lightgray", width=0.5)
    nx.draw_networkx_nodes(roads, coordinates, ax=ax, node_size=6, node_color="lightgray")
    nx.draw_networkx_nodes(roads, coordinates, nodelist=list(expanded), ax=ax, node_size=10, node_color="skyblue")
    nx.draw_networkx_edges(roads, coordinates, edgelist=list(zip(path, path[1:])), ax=ax, edge_color="red", width=2.5)
    nx.draw_networkx_nodes(roads, coordinates, nodelist=[origin, destination], ax=ax, node_size=80, node_color=["green", "gold"])
    ax.set_title(f"{name}\nprohlédnuto {len(expanded)} z {roads.number_of_nodes()} křižovatek, délka trasy {cost:.3f}, {1000 * elapsed:.1f} ms")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

# kontrola proti implementaci v networkx
print("Délka trasy podle networkx A*:", nx.astar_path_length(roads, origin, destination,
                                                            heuristic=lambda u, v: math.dist(coordinates[u], coordinates[v]),
                                                            weight="weight"))

## Kde jinde se A* používá?

| Oblast | Co je *uzel*? | Co je heuristika? |
|---|---|---|
| Počítačové hry (RTS, RPG, roguelike) | dlaždice / polygon navmeshe | vzdušná nebo oktilová vzdálenost |
| Mobilní roboty, marsovská vozítka, autonomní auta | poloha robotu (x, y, natočení) | vzdálenost k cíli bez překážek |
| Skladové AGV, vychystávání | buňka mřížky | manhattanská vzdálenost |
| GPS navigace | křižovatka | vzdušná vzdálenost / čas jízdy max. rychlostí |
| Návrh plošných spojů a čipů (routing) | buňka v routovací vrstvě | manhattanská vzdálenost |
| Automatické plánování (logistika, sondy) | stav světa | relaxovaný problém (ignoruj některá omezení) |
| Hlavolamy (15-puzzle, Rubikova kostka) | konfigurace | pattern databases |
| Rozpoznávání řeči, strojový překlad | částečná věta | odhad zbývajícího skóre |

Recept je vždy stejný: definuj stavy, tahy, ceny – a najdi přípustnou heuristiku *relaxací* problému
(odstraň zdi → vzdušná čára, nech dlaždice létat → manhattanská vzdálenost).

## Cvičení: diagonální pohyb

Dovolte herní jednotce pohyb i po diagonále (8 směrů) s cenou $\sqrt{2}$ za diagonální krok.

1. Naprogramujte `grid_neighbors_8`.
2. Naprogramujte *oktilovou* heuristiku $h = (\Delta_{max} - \Delta_{min}) + \sqrt{2}\,\Delta_{min}$.
3. Je manhattanská heuristika s diagonálami stále přípustná? Ověřte experimentálně: porovnejte cenu cesty
   nalezenou s Manhattanem proti Dijkstrovi (`h = 0`).

In [ ]:
def grid_neighbors_8(grid: np.ndarray):
    rows, cols = grid.shape

    def neighbors(cell):
        r, c = cell
        # YOUR CODE HERE: yield ((nr, nc), cena) pro všech 8 směrů; rovný krok stojí 1, diagonální sqrt(2)
        yield from ()
    return neighbors


def octile(a, b):
    # YOUR CODE HERE
    return 0


# neighbors_8 = grid_neighbors_8(game_map)
# for name, h in [("Dijkstra", zero), ("Manhattan", manhattan), ("Euklid", euclidean), ("Oktil", octile)]:
#     path, cost, expanded = a_star(start, goal, neighbors_8, lambda n, h=h: h(n, goal))
#     print(f"{name:10s} prohlédnuto: {len(expanded):5d}   cena cesty: {cost:.3f}")